# LLM Order Automation Pipeline — Demo

End-to-end demo of the order automation pipeline:
1. Customer email arrives
2. OpenAI extracts product + quantity
3. Pinecone RAG matches against SKU catalog
4. Low-confidence matches flagged for human review
5. Stock checked, auto-reply generated

**Prerequisites:** Fill in `.env` with your OpenAI and Pinecone keys, then run `setup_index.py` once.

In [ ]:
# Install dependencies
!pip install openai pinecone pandas python-dotenv -q

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
from src.pipeline import OrderPipeline
from data.sample_emails import SAMPLE_EMAILS

print(f"Loaded {len(SAMPLE_EMAILS)} sample emails")

## Step 1 — Index Setup (Run Once)
Embeds SKU catalog and upserts into Pinecone. Skip if already done.

In [ ]:
# Run once to set up Pinecone index
# from src.setup_index import upsert_skus
# from pathlib import Path
# upsert_skus(Path('data/sample_skus.csv'))
print("Uncomment above lines to run index setup (needs Pinecone + OpenAI keys)")

## Step 2 — Preview Sample Emails

In [ ]:
for email in SAMPLE_EMAILS[:3]:
    print(f"{'='*60}")
    print(f"ID: {email['email_id']} | From: {email['from']}")
    print(f"Subject: {email['subject']}")
    print(f"Body:\n{email['body'][:300]}...")
    print()

## Step 3 — Preview SKU Catalog

In [ ]:
df_skus = pd.read_csv('data/sample_skus.csv')
print(f"Total SKUs: {len(df_skus)} across {df_skus['category'].nunique()} categories")
df_skus.head(10)

In [ ]:
# SKU distribution by category
df_skus.groupby('category').agg(
    sku_count=('sku_id', 'count'),
    avg_price=('unit_price', 'mean'),
    avg_stock=('stock', 'mean')
).round(2)

## Step 4 — Run Pipeline on a Single Email

In [ ]:
# Initialize pipeline (connects to Pinecone)
pipeline = OrderPipeline()

In [ ]:
# Process a single clear order email
email = SAMPLE_EMAILS[0]  # General Hospital order

result = pipeline.process_email(
    email_id=email['email_id'],
    email_from=email['from'],
    email_subject=email['subject'],
    email_body=email['body'],
    verbose=True,
)

In [ ]:
# Inspect matched items
import pandas as pd
match_df = pd.DataFrame([
    {
        'product_query': m['product_query'],
        'matched_sku': m['sku_id'],
        'matched_name': m['name'],
        'quantity': m['quantity'],
        'stock': m['stock'],
        'similarity': m['similarity_score'],
        'needs_review': m['needs_review'],
    }
    for m in result['matched_items']
])
match_df

## Step 5 — Process Ambiguous Email (Human Review Demo)

In [ ]:
# EMAIL-006 has vague product descriptions — triggers human review
ambiguous_email = SAMPLE_EMAILS[5]
print(f"Email body:\n{ambiguous_email['body']}")

In [ ]:
ambiguous_result = pipeline.process_email(
    email_id=ambiguous_email['email_id'],
    email_from=ambiguous_email['from'],
    email_subject=ambiguous_email['subject'],
    email_body=ambiguous_email['body'],
    verbose=True,
)
print(f"\nItems flagged for review: {len(ambiguous_result['review_items'])}")

## Step 6 — Batch Process All Emails

In [ ]:
all_results = pipeline.process_batch(SAMPLE_EMAILS, verbose=False)

# Summary
summary_df = pd.DataFrame([
    {
        'email_id': r['email_id'],
        'status': r['summary'],
        'items_extracted': len(r['order_items']),
        'auto_processed': len(r['auto_items']),
        'flagged_review': len(r['review_items']),
        'fulfillable': len(r['fulfillable']),
        'out_of_stock': len(r['out_of_stock']),
    }
    for r in all_results
])
print(summary_df.to_string(index=False))

In [ ]:
# Show all generated replies
for r in all_results:
    print(f"\n{'='*60}")
    print(f"{r['email_id']} | Status: {r['summary'].upper()}")
    print('='*60)
    print(r['reply_email'])

## Architecture Reference

```
Outlook Inbox (orders@company.com)
        │
        ▼
Microsoft Graph API Webhook
        │
        ▼
Azure Functions (azure_function/__init__.py)
        │
        ▼
OrderPipeline.process_email()
        │
   ┌────┴─────────────────────┐
   ▼                          ▼
extract_order_items()    (OpenAI GPT-4o-mini)
   │
   ▼
SKUMatcher.match_order_items()  (OpenAI Embeddings + Pinecone)
   │
   ├── High confidence → auto_items → check_stock() → generate_reply()
   │                                                         │
   │                                                         ▼
   │                                               Auto-reply via Graph API
   │
   └── Low confidence  → review_items → Outlook 'Review Queue' folder
                                        (pre-populated draft for human)
```